<a href="https://colab.research.google.com/github/SyameimaruKoa/wd14-tagger-xmp/blob/main/run_colab_server.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# WD14 Tagger Universal - Google Colab サーバー

このノートブックは、Google Colab 上で WD14 Tagger をサーバーモードとして起動し、ローカル環境から接続して画像タグ付けを行うためのものじゃ。
接続のセキュリティ確保と簡便化のために **Tailscale** を使用し、さらに **Google ドライブ連携** により Tailscale 認証状態および接続情報（IP・ホスト名・実行コマンド等）を自動保存・同期できるぞ。

### 🌟 Google ドライブ連携によるシークレット不要の自動再接続モード
Google ドライブ上に Tailscale の認証情報（`tailscaled.state`）を保存することで、**初回のみ認証すれば、2回目以降は Colab のシークレット登録や認証キーの入力なしで即座に自動再接続** できるのじゃ！

## 事前準備（Tailscale 認証キーの作成と接続モードの選択）

本ノートブックでは、Tailscale を使用して Colab とローカル PC 間を安全に接続するのじゃ。
以下の 2 つの利用モードがあるぞ：

---

### A. Google ドライブ保存モード（推奨・デフォルト）
Google ドライブに Tailscale の接続状態を保存するため、**初回に 1 度だけ認証すれば、次回以降はシークレットの登録もキーの入力も一切不要でワンクリック起動** できるぞ。

1. **Tailscale 認証キーの作成**:
   - Tailscale 管理画面の [Settings > Keys](https://login.tailscale.com/admin/settings/keys) にアクセスするのじゃ。
   - **Generate auth key** をクリックし、以下のように設定するのじゃ：
     - **Description**: わかりやすい名前（例: `google-colab`）
     - **Reusable**: **ON (チェックを入れる)**
     - **Ephemeral**: **OFF (チェックを外す)** ★永続化のため必ず OFF にするのじゃ！
     - **Tags**: `tag:Guest` を選択するのじゃ（事前に Tailscale ACL で `tagOwners` 定義が必要）
   - 生成されたキー（`tskey-auth-...`）をコピーするのじゃ。
2. **初回起動時の認証**:
   - ノートブック実行時に、キー入力プロンプトまたは Colab Secrets（`TAILSCALE_AUTHKEY`）から認証するのじゃ。
   - （※キーが手元にない場合でも、プロンプトで Enter を押せばブラウザログイン用 URL が表示され、Web ログインで認証できるぞ）
   - 認証完了後、接続状態が Google ドライブのマイドライブ（`MyDrive/wd14-tagger/tailscaled.state`）に自動保存されるぞ。
3. **次回以降の起動**:
   - Google ドライブから自動的に状態が復元されるため、**シークレット登録やキー入力なしで自動的に接続** されるぞ！

---

### B. 一時利用モード（Google ドライブ保存 OFF）
Google ドライブに保存せず、毎回使い捨てのノードとして起動したい場合：
1. Tailscale 認証キー作成時に **Ephemeral: ON** に設定するのじゃ。
2. Google Colab の Secrets（鍵マーク）に `TAILSCALE_AUTHKEY` を登録するか、セル実行時に入力するのじゃ。

## 1. Environment構築 & リポジトリの準備

プロジェクトのファイルをクローンし、必要なライブラリおよびONNX Runtime（GPUが利用可能ならシステムCUDAバージョンに応じたGPU版、無ければCPU版）をインストールするのじゃ。

In [ ]:
import os, subprocess, re, sys
has_gpu = False
try:
    subprocess.run("nvidia-smi", stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, shell=True, check=True)
    has_gpu = True
except Exception:
    pass
ort_pkg = "onnxruntime"
if has_gpu:
    cuda_major = 12
    try:
        res = subprocess.run(["nvcc", "--version"], capture_output=True, text=True)
        match = re.search(r"release (\d+)\.", res.stdout)
        if match: cuda_major = int(match.group(1))
    except Exception:
        if os.path.exists("/usr/local/cuda"):
            try:
                real_path = os.path.realpath("/usr/local/cuda")
                match = re.search(r"cuda-(\d+)\.", real_path)
                if match: cuda_major = int(match.group(1))
            except Exception: pass
    if cuda_major >= 13:
        ort_pkg = "onnxruntime-gpu"
    elif cuda_major == 12:
        ort_pkg = "onnxruntime-gpu<1.27.0"
    else:
        ort_pkg = "onnxruntime-gpu<1.17.0"
if not os.path.exists("embed_tags_universal.py"):
    !git clone https://github.com/SyameimaruKoa/wd14-tagger-xmp.git
    %cd wd14-tagger-xmp
else:
    !git pull
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt", ort_pkg], check=True)

## 2. Tailscale のインストールと接続

Tailscale をインストールして起動し、接続を行うのじゃ。TailNode名（ホスト名）はデフォルトで `google-colab` に設定されるぞ。
`USE_GOOGLE_DRIVE = True` の場合、Google ドライブに保存された認証状態（`tailscaled.state`）を自動検出し、**シークレットやキー入力不要で即座に再接続** するぞ。
初回実行時（または保存情報がない場合）は、Colab Secrets (`TAILSCALE_AUTHKEY`)、手動入力、またはブラウザ認証リンクから認証し、自動的に Google ドライブへ保存するのじゃ。

In [ ]:
#@title Tailscale のセットアップ & 接続
HOSTNAME = "google-colab" #@param {type:"string"}
USE_GOOGLE_DRIVE = True #@param {type:"boolean"}
DRIVE_FOLDER_NAME = "wd14-tagger" #@param {type:"string"}
RESET_TAILSCALE_STATE = False #@param {type:"boolean"}

import os, subprocess, re, time, json, shutil

# Tailscale のインストール確認
if not shutil.which("tailscale"):
    print("[INFO] Tailscale をインストールしています...")
    res = subprocess.run("curl -fsSL https://tailscale.com/install.sh | sh", shell=True, capture_output=True, text=True)
    if res.returncode != 0:
        print(f"[ERROR] Tailscale のインストールに失敗しました: {res.stderr}")
        raise RuntimeError("Tailscale installation failed.")

os.makedirs("/var/run/tailscale", exist_ok=True)
os.makedirs("/var/lib/tailscale", exist_ok=True)
local_state_file = "/var/lib/tailscale/tailscaled.state"
host_name = HOSTNAME.strip() if HOSTNAME else "google-colab"

# Google ドライブのマウントと保存先フォルダの準備
drive_mounted = False
drive_folder = None
state_file_drive = None

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        if not os.path.exists("/content/drive/MyDrive"):
            print("[INFO] Google ドライブをマウントしています...")
            drive.mount("/content/drive")
        folder_name = DRIVE_FOLDER_NAME.strip("/ ") if DRIVE_FOLDER_NAME else "wd14-tagger"
        drive_folder = os.path.join("/content/drive/MyDrive", folder_name)
        os.makedirs(drive_folder, exist_ok=True)
        drive_mounted = True
        state_file_drive = os.path.join(drive_folder, "tailscaled.state")
        print(f"[INFO] Google ドライブ連携有効: {drive_folder}")
    except Exception as e:
        print(f"[WARN] Google ドライブのマウントに失敗しました: {e}")

# 状態リセット要求がある場合の処理
if RESET_TAILSCALE_STATE:
    if os.path.exists(local_state_file):
        os.remove(local_state_file)
    if state_file_drive and os.path.exists(state_file_drive):
        os.remove(state_file_drive)
    print("[INFO] Tailscale の保存済み認証状態をリセットしました。")

# Google ドライブから認証状態 (tailscaled.state) を復元
has_saved_state = False
if drive_mounted and state_file_drive and os.path.exists(state_file_drive) and not RESET_TAILSCALE_STATE:
    try:
        shutil.copy2(state_file_drive, local_state_file)
        has_saved_state = True
        print(f"[INFO] Google ドライブから Tailscale 認証状態 ({state_file_drive}) を復元しました。")
    except Exception as e:
        print(f"[WARN] 認証状態の復元に失敗しました: {e}")

# 既存の tailscaled を停止して新規起動
subprocess.run(["pkill", "-9", "tailscaled"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(0.5)
subprocess.Popen(["tailscaled", "--tun=userspace-networking"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(30):
    if os.path.exists("/var/run/tailscale/tailscaled.sock"):
        break
    time.sleep(0.3)

# 接続実行
connected = False
if has_saved_state:
    print(f"[INFO] 保存済みの認証情報を使用して Tailscale ({host_name}) に再接続しています（シークレット・キー入力不要）...")
    res = subprocess.run(["tailscale", "up", f"--hostname={host_name}", "--accept-dns=false", "--ssh", "--reset"], capture_output=True, text=True)
    if res.returncode == 0:
        print("[SUCCESS] Tailscale への自動再接続に成功しました！")
        connected = True
    else:
        print(f"[WARN] 保存情報での再接続に失敗しました ({res.stderr.strip()})。再認証を行います...")

if not connected:
    auth = None
    try:
        from google.colab import userdata
        auth = userdata.get("TAILSCALE_AUTHKEY")
        if auth:
            print("[INFO] Colab Secrets から TAILSCALE_AUTHKEY を取得しました。")
    except Exception:
        pass

    if not auth:
        print("\n" + "=" * 60)
        print("【Tailscale 認証】")
        print("Tailscale の認証キー（tskey-auth-...）またはセットアップコマンドを入力してください。")
        print("※キーをお持ちでない場合は、何も入力せず Enter を押すとブラウザ認証リンクが表示されます。")
        print("=" * 60)
        inp = input("Tailscale Auth Key (or press Enter for Web Login): ").strip()
        if inp:
            match = re.search(r"(tskey-(?:auth-)?[a-zA-Z0-9_-]+)", inp)
            auth = match.group(1) if match else inp

    if auth:
        res = subprocess.run(["tailscale", "up", f"--auth-key={auth}", f"--hostname={host_name}", "--accept-dns=false", "--ssh", "--reset"], capture_output=True, text=True)
        if res.returncode != 0:
            print(f"[ERROR] 接続エラー: {res.stderr.strip()}")
            res.check_returncode()
        print("[SUCCESS] 認証キーを使用して Tailscale に接続しました！")
        connected = True
    else:
        print("[INFO] ブラウザ認証を開始します。表示される URL を開いてログインしてください：\n")
        proc = subprocess.Popen(["tailscale", "up", f"--hostname={host_name}", "--accept-dns=false", "--ssh", "--reset"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        while True:
            line = proc.stdout.readline()
            if not line:
                break
            print(line, end="")
            if "Success" in line or "logged in" in line.lower():
                connected = True
                break
        proc.wait()

    # 初回接続成功時、Google ドライブへ認証状態を保存
    if drive_mounted and state_file_drive and os.path.exists(local_state_file):
        try:
            shutil.copy2(local_state_file, state_file_drive)
            print(f"\n[INFO] Tailscale 認証状態を Google ドライブに保存しました: {state_file_drive}")
            print("       次回以降はシークレットやキー入力なしで自動再接続されます！")
        except Exception as e:
            print(f"[WARN] Google ドライブへの認証状態保存に失敗しました: {e}")

# ポート 5000 の serve を設定
subprocess.run(["tailscale", "serve", "--bg", "--tcp", "5000", "5000"], capture_output=True)

# 現在の接続状況を表示
ts_ip = subprocess.run(["tailscale", "ip", "-4"], capture_output=True, text=True).stdout.strip()
print(f"\n[Tailscale Online] IP: {ts_ip} | Hostname: {host_name}")

## 3. 推論サーバーの起動 & Google ドライブへの接続情報保存

推論サーバーをポート 5000 で起動するのじゃ。
`SAVE_TO_GOOGLE_DRIVE = True` にすると、Google ドライブのマイドライブ配下に接続情報（`connection_info.json` / `connection_info.txt`）を自動保存・同期するぞ。
ローカルPC側で Google ドライブアプリを使用している場合、PCから手動でIPやホスト名を入力しなくても自動接続できるようになるのじゃ！

サーバーの動作を終了し、Tailscaleを切断する場合は、セルの左側にある停止ボタン（■）を押すのじゃ。
（※Google ドライブ保存モードでは、次回再接続のために認証情報を保持したまま安全に停止します）

In [ ]:
#@title 推論サーバーの起動 & 接続情報管理
SAVE_TO_GOOGLE_DRIVE = True #@param {type:"boolean"}
DRIVE_FOLDER_NAME = "wd14-tagger" #@param {type:"string"}

import os, sys, subprocess, site, json, asyncio, datetime, shutil
from IPython.display import display, HTML

# Google ドライブのマウント確認と保存先準備
drive_mounted = False
drive_info_dir = None
state_file_drive = None
local_state_file = "/var/lib/tailscale/tailscaled.state"
drive_badge_html = ""

if SAVE_TO_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        if not os.path.exists("/content/drive/MyDrive"):
            print("[INFO] Google ドライブをマウントしています...")
            drive.mount("/content/drive")
        folder_name = DRIVE_FOLDER_NAME.strip("/ ") if DRIVE_FOLDER_NAME else "wd14-tagger"
        drive_info_dir = os.path.join("/content/drive/MyDrive", folder_name)
        os.makedirs(drive_info_dir, exist_ok=True)
        drive_mounted = True
        state_file_drive = os.path.join(drive_info_dir, "tailscaled.state")
        print(f"[INFO] Google ドライブ連携有効: {drive_info_dir}")
        drive_badge_html = f'<div style="display: inline-flex; align-items: center; background: rgba(166, 227, 161, 0.15); border: 1px solid #a6e3a1; color: #a6e3a1; padding: 4px 10px; border-radius: 6px; font-size: 11px; font-weight: 600;">💾 Google Drive 保存完了: MyDrive/{folder_name}/connection_info.json</div>'
    except Exception as e:
        print(f"[WARN] Google ドライブのマウントに失敗しました: {e}")
        drive_badge_html = f'<div style="display: inline-flex; align-items: center; background: rgba(243, 139, 168, 0.15); border: 1px solid #f38ba8; color: #f38ba8; padding: 4px 10px; border-radius: 6px; font-size: 11px;">⚠️ Google Drive 保存失敗: {e}</div>'
else:
    drive_badge_html = '<div style="display: inline-flex; align-items: center; background: rgba(108, 112, 134, 0.2); border: 1px solid #6c7086; color: #a6adc8; padding: 4px 10px; border-radius: 6px; font-size: 11px;">📁 Google Drive 保存: OFF</div>'

# Tailscale IP & ホスト名取得
ip = subprocess.run(["tailscale", "ip", "-4"], capture_output=True, text=True).stdout.strip()
hostname = "google-colab"
try:
    status_raw = subprocess.run(["tailscale", "status", "--json"], capture_output=True, text=True).stdout
    status = json.loads(status_raw)
    hostname = status.get("Self", {}).get("HostName", hostname)
except Exception:
    pass

now_iso = datetime.datetime.now(datetime.timezone.utc).isoformat()
now_str = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# コマンド文字列
ps_cmd_drive = '.\\run_tagger.ps1 -Client -Drive -Path \"C:\\Images\" -Organize'
ps_cmd_magic = f'.\\run_tagger.ps1 -Client -HostIP \"{hostname}\" -Path \"C:\\Images\" -Organize'
ps_cmd_ip = f'.\\run_tagger.ps1 -Client -HostIP \"{ip}\" -Path \"C:\\Images\" -Organize'

sh_cmd_drive = './run_tagger.sh --client --drive -p \"/path/to/images\" --organize'
sh_cmd_magic = f'./run_tagger.sh --client --host \"{hostname}\" -p \"/path/to/images\" --organize'
sh_cmd_ip = f'./run_tagger.sh --client --host \"{ip}\" -p \"/path/to/images\" --organize'

# 接続情報を Google ドライブに書き込み
if drive_mounted and drive_info_dir:
    info_data = {
        "status": "online",
        "server_type": "colab",
        "tailscale_ip": ip,
        "hostname": hostname,
        "port": 5000,
        "url_magicdns": f"http://{hostname}:5000",
        "url_ip": f"http://{ip}:5000",
        "started_at": now_iso,
        "updated_at": now_iso,
        "commands": {
            "powershell_drive": ps_cmd_drive,
            "powershell_magicdns": ps_cmd_magic,
            "powershell_ip": ps_cmd_ip,
            "bash_drive": sh_cmd_drive,
            "bash_magicdns": sh_cmd_magic,
            "bash_ip": sh_cmd_ip
        }
    }
    json_file_path = os.path.join(drive_info_dir, "connection_info.json")
    txt_file_path = os.path.join(drive_info_dir, "connection_info.txt")
    try:
        with open(json_file_path, "w", encoding="utf-8") as f:
            json.dump(info_data, f, indent=4, ensure_ascii=False)
        txt_content = f"""==================================================
WD14 Tagger Server - Connection Info
==================================================
[Server Status]      Online
[Started At]         {now_str}
[Tailscale IP]       {ip}
[MagicDNS Hostname]  {hostname}
[Port]               5000

--------------------------------------------------
■ Connection Commands
--------------------------------------------------

● Google Drive Auto-Sync (Recommended):
  [Windows (PowerShell)]
    {ps_cmd_drive}
  [Linux / macOS (Bash)]
    {sh_cmd_drive}

● MagicDNS Hostname:
  [Windows (PowerShell)]
    {ps_cmd_magic}
  [Linux / macOS (Bash)]
    {sh_cmd_magic}

● Direct Tailscale IP:
  [Windows (PowerShell)]
    {ps_cmd_ip}
  [Linux / macOS (Bash)]
    {sh_cmd_ip}
==================================================
"""
        with open(txt_file_path, "w", encoding="utf-8") as f:
            f.write(txt_content)
        print(f"[INFO] 接続情報ファイルを保存しました: {json_file_path}")
    except Exception as e:
        print(f"[WARN] 接続情報ファイルの保存に失敗しました: {e}")

html_code = f"""
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif; background: #1e1e2e; color: #cdd6f4; padding: 20px; border-radius: 12px; max-width: 750px; box-shadow: 0 4px 20px rgba(0,0,0,0.25); margin: 15px 0; border: 1px solid #313244;">
  <div style="display: flex; align-items: center; justify-content: space-between; border-bottom: 1px solid #313244; padding-bottom: 12px; margin-bottom: 16px;">
    <div style="display: flex; align-items: center;">
      <span style="display: inline-block; width: 10px; height: 10px; background-color: #a6e3a1; border-radius: 50%; margin-right: 8px; box-shadow: 0 0 8px #a6e3a1; animation: pulse 1.8s infinite;"></span>
      <span style="font-size: 16px; font-weight: 600; color: #a6e3a1;">WD14 Tagger Server Active</span>
    </div>
    <div style="display: flex; gap: 8px; align-items: center;">
      <span style="font-size: 11px; background: #313244; color: #cdd6f4; padding: 3px 8px; border-radius: 4px; font-weight: 500;">Port: 5000</span>
    </div>
  </div>
  <div style="margin-bottom: 16px;">
    {drive_badge_html}
  </div>
  <div style="background: rgba(249, 226, 175, 0.1); border: 1px solid #f9e2af; padding: 12px; border-radius: 8px; margin-bottom: 16px; font-size: 12px; color: #f9e2af; line-height: 1.5;">
    ⚠️ <strong>注意:</strong> 起動後、初回接続時は Hugging Face からモデルファイルを自動ダウンロードするため、ロード完了まで 1〜2分 ほどかかります。下に「<code>[INFO] サーバー稼働中 Port: 5000</code>」と表示されるまで、ローカルからの接続はお待ちください。
  </div>
  <div style="display: flex; gap: 12px; margin-bottom: 16px;">
    <div onclick="navigator.clipboard.writeText('{ip}'); alert('Copied Tailscale IP to clipboard: {ip}');" style="flex: 1; background: #11111b; padding: 12px; border-radius: 8px; border: 1px solid #313244; cursor: pointer; position: relative; transition: all 0.2s;" onmouseover="this.style.borderColor='#89b4fa'; this.style.background='#181825';" onmouseout="this.style.borderColor='#313244'; this.style.background='#11111b';">
      <div style="font-size: 10px; color: #a6adc8; text-transform: uppercase; margin-bottom: 4px; font-weight: bold; display: flex; justify-content: space-between;">
        <span>Tailscale IP</span>
        <span style="color: #89b4fa; font-size: 9px;">Click to Copy</span>
      </div>
      <div style="font-family: monospace; font-size: 14px; color: #89b4fa; word-break: break-all;">{ip}</div>
    </div>
    <div onclick="navigator.clipboard.writeText('{hostname}'); alert('Copied MagicDNS Hostname to clipboard: {hostname}');" style="flex: 1; background: #11111b; padding: 12px; border-radius: 8px; border: 1px solid #313244; cursor: pointer; position: relative; transition: all 0.2s;" onmouseover="this.style.borderColor='#f9e2af'; this.style.background='#181825';" onmouseout="this.style.borderColor='#313244'; this.style.background='#11111b';">
      <div style="font-size: 10px; color: #a6adc8; text-transform: uppercase; margin-bottom: 4px; font-weight: bold; display: flex; justify-content: space-between;">
        <span>MagicDNS Hostname</span>
        <span style="color: #f9e2af; font-size: 9px;">Click to Copy</span>
      </div>
      <div style="font-family: monospace; font-size: 14px; color: #f9e2af; word-break: break-all;">{hostname}</div>
    </div>
  </div>
  <div style="margin-bottom: 16px;">
    <h4 style="margin: 0 0 10px 0; font-size: 13px; color: #bac2de; font-weight: 600;">Connection Instructions:</h4>
    <div style="margin-bottom: 12px; background: #181825; padding: 12px; border-radius: 8px; border: 1px solid #313244;">
      <div style="font-size: 12px; color: #89b4fa; font-weight: bold; margin-bottom: 6px;">PowerShell (Windows)</div>
      <div style="font-size: 11px; color: #cba6f7; margin-bottom: 4px;">★ Google Drive 自動同期モード (推奨・入力不要)</div>
      <pre style="background: #11111b; padding: 8px; border-radius: 6px; font-size: 11px; font-family: monospace; overflow-x: auto; margin: 0 0 10px 0; color: #cdd6f4; border: 1px solid #313244; white-space: pre-wrap; word-break: break-all;">{ps_cmd_drive}</pre>
      <div style="font-size: 11px; color: #a6e3a1; margin-bottom: 4px;">● MagicDNS</div>
      <pre style="background: #11111b; padding: 8px; border-radius: 6px; font-size: 11px; font-family: monospace; overflow-x: auto; margin: 0 0 10px 0; color: #cdd6f4; border: 1px solid #313244; white-space: pre-wrap; word-break: break-all;">{ps_cmd_magic}</pre>
      <div style="font-size: 11px; color: #f9e2af; margin-bottom: 4px;">▲ Direct IP</div>
      <pre style="background: #11111b; padding: 8px; border-radius: 6px; font-size: 11px; font-family: monospace; overflow-x: auto; margin: 0; color: #cdd6f4; border: 1px solid #313244; white-space: pre-wrap; word-break: break-all;">{ps_cmd_ip}</pre>
    </div>
    <div style="background: #181825; padding: 12px; border-radius: 8px; border: 1px solid #313244;">
      <div style="font-size: 12px; color: #f9e2af; font-weight: bold; margin-bottom: 6px;">Bash (Linux / macOS)</div>
      <div style="font-size: 11px; color: #cba6f7; margin-bottom: 4px;">★ Google Drive 自動同期モード (推奨・入力不要)</div>
      <pre style="background: #11111b; padding: 8px; border-radius: 6px; font-size: 11px; font-family: monospace; overflow-x: auto; margin: 0 0 10px 0; color: #cdd6f4; border: 1px solid #313244; white-space: pre-wrap; word-break: break-all;">{sh_cmd_drive}</pre>
      <div style="font-size: 11px; color: #a6e3a1; margin-bottom: 4px;">● MagicDNS</div>
      <pre style="background: #11111b; padding: 8px; border-radius: 6px; font-size: 11px; font-family: monospace; overflow-x: auto; margin: 0 0 10px 0; color: #cdd6f4; border: 1px solid #313244; white-space: pre-wrap; word-break: break-all;">{sh_cmd_magic}</pre>
      <div style="font-size: 11px; color: #f9e2af; margin-bottom: 4px;">▲ Direct IP</div>
      <pre style="background: #11111b; padding: 8px; border-radius: 6px; font-size: 11px; font-family: monospace; overflow-x: auto; margin: 0; color: #cdd6f4; border: 1px solid #313244; white-space: pre-wrap; word-break: break-all;">{sh_cmd_ip}</pre>
    </div>
  </div>
  <div style="border-top: 1px solid #313244; padding-top: 12px; text-align: center; font-size: 12px; color: #bac2de;">
    サーバーを停止するには、左のセルの<strong>停止ボタン（■）</strong>を押してください。自動的に接続情報がオフライン化され、安全に切断されます。
  </div>
</div>
<style>
@keyframes pulse {{
  0% {{ transform: scale(0.95); box-shadow: 0 0 0 0 rgba(166, 227, 161, 0.7); }}
  70% {{ transform: scale(1); box-shadow: 0 0 0 6px rgba(166, 227, 161, 0); }}
  100% {{ transform: scale(0.95); box-shadow: 0 0 0 0 rgba(166, 227, 161, 0); }}
}}
</style>
"""
has_gpu = False
try:
    subprocess.run("nvidia-smi", stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, shell=True, check=True)
    has_gpu = True
except Exception:
    pass
args = [sys.executable, "-u", "embed_tags_universal.py", "--mode", "server", "--port", "5000"]
if has_gpu: args.append("--gpu")
env = os.environ.copy()
nvidia_libs = []
for p in site.getsitepackages():
    nvidia_dir = os.path.join(p, "nvidia")
    if os.path.exists(nvidia_dir):
        for root, dirs, files in os.walk(nvidia_dir):
            if "lib" in dirs: nvidia_libs.append(os.path.join(root, "lib"))
cuda_paths = ["/usr/local/cuda/lib64", "/usr/local/nvidia/lib64"] + nvidia_libs
env["LD_LIBRARY_PATH"] = ":".join(cuda_paths) + ":" + env.get("LD_LIBRARY_PATH", "")
process = await asyncio.create_subprocess_exec(*args, env=env, stdout=asyncio.subprocess.PIPE, stderr=asyncio.subprocess.STDOUT)
display(HTML(html_code))
try:
    while True:
        line = await process.stdout.readline()
        if not line:
            break
        sys.stdout.write(line.decode("utf-8"))
        sys.stdout.flush()
except BaseException as e:
    print(f"\n[INFO] 終了シグナルを受信しました ({type(e).__name__})。クリーンアップを実行します...")
    try:
        process.terminate()
    except Exception:
        pass
finally:
    await process.wait()
    if drive_mounted and drive_info_dir:
        try:
            offline_iso = datetime.datetime.now(datetime.timezone.utc).isoformat()
            offline_data = {
                "status": "offline",
                "server_type": "colab",
                "tailscale_ip": ip,
                "hostname": hostname,
                "port": 5000,
                "started_at": now_iso,
                "stopped_at": offline_iso,
                "updated_at": offline_iso
            }
            json_file_path = os.path.join(drive_info_dir, "connection_info.json")
            with open(json_file_path, "w", encoding="utf-8") as f:
                json.dump(offline_data, f, indent=4, ensure_ascii=False)
            print(f"[INFO] Google ドライブの接続情報を offline に更新しました。")
            if state_file_drive and os.path.exists(local_state_file):
                shutil.copy2(local_state_file, state_file_drive)
                print("[INFO] Tailscale の最新認証状態を Google ドライブにバックアップ保存しました。")
        except Exception as ex:
            print(f"[WARN] オフライン状態の更新に失敗しました: {ex}")
        subprocess.run(["tailscale", "down"], capture_output=True)
        print("[INFO] Tailscale を安全に切断しました（次回起動時に再接続できます）。")
    else:
        subprocess.run(["tailscale", "logout"], capture_output=True)
        print("[INFO] Tailscale からログアウトし、安全に終了しました。")

## 4. 終了処理 & 強制ログアウト

※上の起動セルの実行を終了した（または停止ボタンを押した）段階で、自動的にGoogleドライブの接続情報がオフライン化され、安全に切断されます。
通常は手動で以下のセルを実行する必要はありません。
Tailscale のデバイス一覧から完全にログアウト・削除したい場合のみ実行してください。

In [ ]:
#@title Tailscale の完全ログアウト（必要な場合のみ）
!tailscale logout
print("[INFO] Tailscale からログアウトしました。")